# Indexation of spots data set.
**Using Class spotsSet and play with spots considered for indexation and refinement**

### This Notebook is a part of Tutorials on LaueTools Suite. Author:J.-S. Micha  Date: July 2019

In [ ]:
%matplotlib inline

import matplotlib
import numpy as np
import matplotlib.pyplot as plt
import time,copy,os

# third party LaueTools import
import LaueTools.readmccd as RMCCD
import LaueTools.LaueGeometry as F2TC
import LaueTools.indexingSpotsSet as ISS
import LaueTools.IOLaueTools as RWASCII

refinement from guessed solutions with two materials (see script IndexingTwinsSeries)

#### Let's take a simple example of a single Laue Pattern. From the peak search we get 83 spots

In [ ]:
folder= '../Examples/Ge/'
datfilename='Ge0001.dat'

In [ ]:
key_material='Ge'
emin, emax= 5,23

In [ ]:
# detector geometry and parameters as read from Ge0001.det
calibration_parameters = [69.179, 1050.81, 1115.59, 0.104, -0.273]
CCDCalibdict = {}
CCDCalibdict['CCDCalibParameters'] = calibration_parameters
CCDCalibdict['framedim'] = (2048, 2048)
CCDCalibdict['detectordiameter'] = 165.
CCDCalibdict['kf_direction'] = 'Z>0'
CCDCalibdict['xpixelsize'] = 0.08057
# CCDCalibdict can also be simply build by reading the proper .det file
print("reading geometry calibration file")
CCDCalibdict=RWASCII.readCalib_det_file(os.path.join(folder,'Ge0001.det'))
CCDCalibdict['kf_direction'] = 'Z>0'

###### Compute scattering angles from spots pixel positions and detector geometry. Write a .cor file from .dat including these new infos

In [ ]:
F2TC.convert2corfile(datfilename,
                         calibration_parameters,
                         dirname_in=folder,
                        dirname_out=folder,
                        CCDCalibdict=CCDCalibdict)
corfilename = datfilename.split('.')[0] + '.cor'
fullpathcorfile = os.path.join(folder,corfilename)

###### Create an instance of the class spotset. Initialize spots properties to data contained in .cor file

In [ ]:
DataSet = ISS.spotsset()

DataSet.importdatafromfile(fullpathcorfile)

###### Class methods and attributes rely on a dictionnary of spots properties. key = exprimental spot index, val = spots properties 

In [ ]:
[DataSet.indexed_spots_dict[k] for k in range(10)]

In [ ]:
DataSet.getUnIndexedSpotsallData()[:3]

In [ ]:
dict_loop = {'MATCHINGRATE_THRESHOLD_IAL': 100,
                   'MATCHINGRATE_ANGLE_TOL': 0.2,
                   'NBMAXPROBED': 6,
                   'central spots indices': [0,],
                   'AngleTolLUT': 0.5,
                   'UseIntensityWeights': False,
                   'nbSpotsToIndex':10000,
                   'list matching tol angles':[0.5,0.5,0.2,0.2],
                   'nlutmax':3,
                   'MinimumNumberMatches': 3,
                   'MinimumMatchingRate':3
                   }
grainindex=0
DataSet = ISS.spotsset()
    
DataSet.pixelsize = CCDCalibdict['xpixelsize']
DataSet.dim = CCDCalibdict['framedim']
DataSet.detectordiameter = CCDCalibdict['detectordiameter']
DataSet.kf_direction = CCDCalibdict['kf_direction']
DataSet.key_material = key_material
DataSet.emin = emin
DataSet.emax = emax


##### Normally we read all spots data from a .cor file

In [ ]:
DataSet.importdatafromfile(fullpathcorfile)
DataSet.emin

##### but we can import a custom list of spots. For example, starting from spots a the previous .cor file

In [ ]:
Gespots = RWASCII.readfile_cor(fullpathcorfile)[0]

In [ ]:
# 2theta chi X, Y Intensity of the first 7 spots
Gespots[:7,:5]

In [ ]:
tth,chi,X,Y,I=Gespots[:,:5].T
exp_data_all=np.array([tth,chi,I,X,Y])
exp_data_all.shape

In [ ]:
#select some exp spots from absolute index   (6,0,2,30,9,8,20,10,5,1,7,14)
tth_e,chi_e,X_e,Y_e,I_e = (np.take(Gespots[:,:5],(6,0,2,30,9,8,20,10,5,1,7,14),axis=0)).T
exp_data=np.array([tth_e,chi_e,I_e,X_e,Y_e])

#### spots data must be imported as an array of 5 elements: 2theta, chi, Intensity, pixelX, pixelY

In [ ]:
DataSet.importdata(exp_data)
DataSet.detectorparameters = calibration_parameters
DataSet.nbspots = len(exp_data[0])
DataSet.filename = 'short_'+corfilename
#DataSet.setSelectedExpSpotsData(0)
DataSet.getSelectedExpSpotsData(0)

### core function to index a set of spots

by defaut DataSet.getUnIndexedSpotsallData() is called

if use_file = 0, then current non indexed exp. spots will be considered for indexation

if use_file  = 1, reimport data from file and reset also spots properties dictionary (i.e. with status unindexed)

In [ ]:
DataSet.IndexSpotsSet(fullpathcorfile, key_material, emin, emax, dict_loop, None,
                         use_file=0, # if 1, reimport data from file and reset also spots properties dictionary
                         IMM=False,LUT=None,n_LUT=dict_loop['nlutmax'],angletol_list=dict_loop['list matching tol angles'],
                        nbGrainstoFind=1,
                      starting_grainindex=0,
                      MatchingRate_List=[1, 1, 1,1,1,1,1,1],
                        verbose=0, previousResults=None,
                        corfilename=corfilename)

In [ ]:
index_grain_retrieve=0
print("number of indexed spots", len(DataSet.getallIndexedSpotsallData()[index_grain_retrieve]))

### Results of indexation can be found in attributes or through methods

In [ ]:
spotsdata=DataSet.getSummaryallData()
print("first 2 indexed spots properties\n")
print('#spot  #grain 2theta chi X Y I h k l Energy')
print(spotsdata[:2])

In [ ]:
print('#grain  : [Npairs = Nb pairs with tolerance angle %.4f, 100*Npairs/Ndirections theo.]'%dict_loop['list matching tol angles'][-1])
DataSet.dict_grain_matching_rate


In [ ]:
print("#grain  : deviatoric strain")
DataSet.dict_grain_devstrain

In [ ]:
#RefinedUB= DataSet.dict_grain_matrix
print("#grain  : refined UB matrix")
DataSet.dict_grain_matrix



In [ ]:
print([DataSet.indexed_spots_dict[k] for k in range(10)])